#### Appending Multiple CSV Files


In [9]:
import pandas as pd
import numpy as np
import glob
import os
import arcpy


In [10]:
### Set up path to Geodatabase

gdb = r"C:\Users\beste\Desktop\ArcGIS_Pro\Portfolio\PowerOutages\01 Data\MN_Tornado_Outage.gdb"
csv_folder= r"C:\Users\beste\Desktop\ArcGIS_Pro\Portfolio\PowerOutages\01 Data\eaglei_outages"

### Set Environment
arcpy.env.workspace = gdb 
arcpy.env.overwriteOutput= True



In [ ]:
### Move through the data folder, Print file names 
for foldername, folders, filenames in arcpy.da.Walk(csv_folder):
        for file in filenames:
            print(file)

            # for the power outage data , print field names
            if file.startswith("eaglei"):
                file =os.path.join(csv_folder,file)
                df= pd.read_csv(file)
                print(df.columns)

coverage_history.csv
eaglei_outages_2014.csv
Index(['fips_code', 'county', 'state', 'sum', 'run_start_time'], dtype='object')
eaglei_outages_2015.csv
Index(['fips_code', 'county', 'state', 'sum', 'run_start_time'], dtype='object')
eaglei_outages_2016.csv
Index(['fips_code', 'county', 'state', 'sum', 'run_start_time'], dtype='object')
eaglei_outages_2017.csv
Index(['fips_code', 'county', 'state', 'sum', 'run_start_time'], dtype='object')
eaglei_outages_2018.csv
Index(['fips_code', 'county', 'state', 'sum', 'run_start_time'], dtype='object')
eaglei_outages_2019.csv
Index(['fips_code', 'county', 'state', 'sum', 'run_start_time'], dtype='object')
eaglei_outages_2020.csv
Index(['fips_code', 'county', 'state', 'sum', 'run_start_time'], dtype='object')
eaglei_outages_2021.csv
Index(['fips_code', 'county', 'state', 'sum', 'run_start_time'], dtype='object')
eaglei_outages_2022.csv
Index(['fips_code', 'county', 'state', 'sum', 'run_start_time'], dtype='object')


In [36]:
def log(msg):
    arcpy.AddMessage(msg)          # works inside a GP tool
    print(msg)                     # also shows in stand-alone script

### **Add a New Field**
#### `arcpy.management.AddField()`
- use to add new fields to a feature class attributes
- arcpy.management.AddField(input_data, field_name, field_type, {field_alias},{field_length} , only applies to text--Default is 255, {template})
    - field types include: SHORT, DOUBLE, TEXT , DATE, DATEONLY, TIMEONLY, BLOB, RASTER


In [48]:
### ======================= parameters

# create output table

gdb = r"C:\Users\beste\Desktop\ArcGIS_Pro\Portfolio\PowerOutages\01 Data\MN_Tornado_Outage.gdb"
out_table_name = "power_outages_2014_2022"
csv_folder= r"C:\Users\beste\Desktop\ArcGIS_Pro\Portfolio\PowerOutages\01 Data\eaglei_outages"
out_table = os.path.join(gdb,out_table_name)

# Set Environment
arcpy.env.workspace = gdb 
arcpy.env.overwriteOutput= True

keep_cols  = ["fips_code", "county", "state", "sum", "run_start_time"]


if not arcpy.Exists(out_table):
    arcpy.management.CreateTable(gdb, os.path.basename(out_table))
    arcpy.management.AddField(out_table, "fips_code",     "TEXT", 5)
    arcpy.management.AddField(out_table, "county",        "TEXT", 30)
    arcpy.management.AddField(out_table, "state",         "TEXT", 30)
    arcpy.management.AddField(out_table, "customers_out", "LONG")
    arcpy.management.AddField(out_table, "run_start_time",     "DATE")
    arcpy.management.AddIndex(out_table, "fips_code", "idx_fips")
    arcpy.management.AddIndex(out_table, "run_start_time",  "idx_time")

schema     = ["fips_code", "county", "state", "customers_out", "run_start_time"]

np_dtype   = [('fips_code','<U5'),
              ('county','<U30'),
              ('state','<U30'),
              ('customers_out','<i4'),
              ('run_start_time','M8[ms]')]
chunksize  = 500_000

for csv in sorted(glob.glob(os.path.join(csv_folder, "eaglei_outages_*.csv"))):
    log(f"file name {os.path.basename(csv)}")

    ### Read the csv into a dataframe in chunks of 500K rows 
    # Enumerate used to track chunks and mark the fields of the first chunk for comparison to the target table
    for i, df in enumerate(pd.read_csv(csv,
                                       chunksize=chunksize,
                                       usecols=keep_cols,
                                       dtype={"fips_code":"string"}), 1): # 1, means enumerate starts the counter at 1 instead of 0
        # MN only
        df = df[df["fips_code"].str.startswith("27")]
        if df.empty:
            continue

        # clean
        df = (df.rename(columns={
                                 "sum":"customers_out"})
                .dropna(subset=["customers_out"])
                .assign(fips_code     =lambda d: d.fips_code.str.zfill(5),
                        customers_out =lambda d: d.customers_out.astype("int32"),
                        run_start_time     =lambda d:
                                        pd.to_datetime(d.run_start_time, utc=True)
                                          .dt.tz_localize(None))
                .drop_duplicates(subset=["fips_code","run_start_time"]))

        log(f"chunk {i}: {len(df):,} rows after filter")

        # DataFrame → NumPy array
        arr = np.rec.fromrecords(df[schema].to_records(index=False),
                                 dtype=np_dtype)

        ###============== NumPy array to a tmp FGDB table
  
        tmp = r"in_memory\tmp" # define a temp path for intermediate table output in RAM
        if arcpy.Exists(tmp):
            arcpy.management.Delete(tmp) # remove any leftover table if it exists


        arcpy.da.NumPyArrayToTable(arr, tmp) 

        # **Debug: show field lists once**
        if i == 1:
            log("   tmp fields:  " + ", ".join(f.name for f in arcpy.ListFields(tmp)))
            log("   dest fields: " + ", ".join(f.name for f in arcpy.ListFields(out_table)))

        # Append with schema CHECK
        arcpy.management.Append(tmp, out_table, "TEST") # forces fields to match otherwise throws an error
        arcpy.management.Delete(tmp)

        log(f"   chunk {i}: appended ✔")

log(" All files processed")

file name eaglei_outages_2014.csv
chunk 1: 8,934 rows after filter
   tmp fields:  OBJECTID, fips_code, county, state, customers_out, run_start_time
   dest fields: OBJECTID, fips_code, county, state, customers_out, run_start_time
   chunk 1: appended ✔
chunk 2: 7,611 rows after filter
   chunk 2: appended ✔
chunk 3: 14,467 rows after filter
   chunk 3: appended ✔
chunk 4: 6,510 rows after filter
   chunk 4: appended ✔
file name eaglei_outages_2015.csv
chunk 1: 14,586 rows after filter
   tmp fields:  OBJECTID, fips_code, county, state, customers_out, run_start_time
   dest fields: OBJECTID, fips_code, county, state, customers_out, run_start_time
   chunk 1: appended ✔
chunk 2: 16,559 rows after filter
   chunk 2: appended ✔
chunk 3: 13,003 rows after filter
   chunk 3: appended ✔
chunk 4: 13,888 rows after filter
   chunk 4: appended ✔
chunk 5: 16,425 rows after filter
   chunk 5: appended ✔
chunk 6: 12,363 rows after filter
   chunk 6: appended ✔
chunk 7: 12,220 rows after filter
   

### **Append Tables**
#### `arcpy.management.Append()`
- Adds new features to an **existing dataset**
- Can add point, line or polygon feature classes to an **existing dataset of the same type** ( e.g., cannot add line fc to a point fc)
- Can also add several tables to an existing table or several rasters to an existing raster
- arcpy.management.Append(inputs, target, {schema_type}, {field_mapping}, {subtype}, {expression}, {match_fields}, {update_geometry}, {enforce_domains})
- key fields: 
    - input: data to be appended to the target dataset (Note you can combine Tables and Feature Classes e.g., Fc+Table--> NewTable with Fc attributes)
    - target: existing dataset where the input will be added
    - schema_type: whether fields must match. TEST, causes error if fields do not match. NO_TEST allows fields to be null,TEST_AND_SKIP, will leave out the non-matching fields

In [50]:
# 1. add the new field once
arcpy.management.AddField("mn_outages_poly",
                          "pct_out",               # name
                          "DOUBLE",                # type
                          field_alias="Percent_Out")

<Result 'C:\\Users\\beste\\Desktop\\ArcGIS_Pro\\Portfolio\\PowerOutages\\01 Data\\MN_Tornado_Outage.gdb\\mn_outages_poly'>

In [51]:
# 2. populate it (% customers without power)
expr = "!customers_out! / !customers_tracked! * 100"   # adjust names if needed
arcpy.management.CalculateField("mn_outages_poly",
                                "pct_out",
                                expr,
                                "PYTHON3")

ExecuteError: ERROR 000539: Invalid field customers_tracked
Failed to execute (CalculateField).
